In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost
import sklearn

I have decided to work on the hourly dataset :

In [2]:
df=pd.read_csv('/home/yazid/sklearn/bikes/data/hour.csv')

In [3]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [4]:
len(df)

17379

In [5]:
df.drop('instant',axis=1,inplace=True)

In [6]:
df['dteday']=pd.to_datetime(df['dteday'])

Let's create some features :

Cyclical temporical features :

In [7]:
df['hr_sin']=np.sin(2*np.pi*(df['hr']/24))
df['hr_cos']=np.cos(2*np.pi*(df['hr']/24))
df['mnth_sin']=np.sin(2*np.pi*(df['mnth']/12))
df['mnth_cos']=np.cos(2*np.pi*(df['mnth']/12))
df['week_sin']=np.sin(2*np.pi*(df['weekday']/7))
df['week_cos']=np.cos(2*np.pi*(df['weekday']/7))

Commute features :

In [8]:
df['Rush_Hour']=((df['workingday']==1)&(df['hr'].isin([7,8,9,17,18,19]))).astype(bool)
df['Off_Peak']=(df['hr'].isin([0,1,2,3,4,5])).astype(bool)

In [9]:
df['Previous_hour_cnt']=df['cnt'].shift(1)
df['Previous_day_cnt']=df['cnt'].shift(24)

We create now weather based columns :

In [10]:
df['delta_temp']=df['temp']-df['atemp']
df['temp*hum']=df['temp']*df['hum']
df['bad_weather']=(df['weathersit']>=3).astype(bool)
df['windspeed*temp']=(df['temp']*df['windspeed'])

Let's create rolling averages feature:

In [11]:
df['cnt_roll_3h_mean']=(df['cnt'].shift(1).rolling(window=3).mean())
df['cnt_roll_3h_std']=(df['cnt'].shift(1).rolling(window=3).std())
df['cnt_roll_24_std']=(df['cnt'].shift(1).rolling(window=24).std())

In [12]:
#We log transform target since it is righly skewed:
df['log_cnt']=np.log1p(df['cnt'])
df.drop('cnt',axis=1,inplace=True)

Let's create function to extract feature_importance:

In [13]:
def feature_importance(model):
    feature_names=model.feature_names_in_
    return pd.DataFrame({'Feature':feature_names,
                        'Importance':model.feature_importances_}).sort_values('Importance',ascending=False).reset_index(drop=True)

In [14]:
df.dropna(inplace=True)

In [15]:
#This will be used later for train_test_split:
len(df)*0.8

13884.0

In [16]:
X=df.drop(['log_cnt','casual','registered','dteday'],axis=1)
y=df['log_cnt']
X_train=X[:13768]
X_test=X[13768:]
y_train=y[:13768]
y_test=y[13768:]

In [17]:
from sklearn.ensemble import RandomForestRegressor
v0=RandomForestRegressor()
v0.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the numb

In [18]:
from sklearn.metrics import mean_absolute_error
y_pred_0=v0.predict(X_test)
mae_0=mean_absolute_error(y_test,y_pred_0 )
(mae_0 /y_test.median())*100

np.float64(3.4096302115136377)

Our first model is performing well let's try to improve things further with RandomizedSearchCV

In [19]:
'''
# I ran this on colab gpu and got results i wanted:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit


tscv=TimeSeriesSplit(n_splits=5)
xgb_model = XGBRegressor(
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [6, 8, 10,12],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree':[0.6,0.8,1.0],
    'gamma':[0,0.1,0.2]
}

randomized_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_grid,
    n_iter=40,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    verbose=1
)

randomized_search.fit(X_train, y_train)
'''

'\n# I ran this on colab gpu and got results i wanted:\nfrom xgboost import XGBRegressor\nfrom sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit\n\n\ntscv=TimeSeriesSplit(n_splits=5)\nxgb_model = XGBRegressor(\n    tree_method="hist",\n    device="cuda",\n    random_state=42\n)\n\nxgb_grid = {\n    \'n_estimators\': [100, 300, 500],\n    \'learning_rate\': [0.01, 0.05, 0.1],\n    \'max_depth\': [6, 8, 10,12],\n    \'subsample\': [0.6, 0.8, 1.0],\n    \'colsample_bytree\':[0.6,0.8,1.0],\n    \'gamma\':[0,0.1,0.2]\n}\n\nrandomized_search = RandomizedSearchCV(\n    estimator=xgb_model,\n    param_distributions=xgb_grid,\n    n_iter=40,\n    cv=tscv,\n    scoring=\'neg_root_mean_squared_error\',\n    random_state=42,\n    verbose=1\n)\n\nrandomized_search.fit(X_train, y_train)\n'

In [20]:
'''
Here are results from colab_gpu:
print(randomized_search.best_params_)
#{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}
'''

"\nHere are results from colab_gpu:\nprint(randomized_search.best_params_)\n#{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}\n"

In [21]:
from xgboost import XGBRegressor
v1=XGBRegressor(n_esimators=500,subsample=0.8,max_depth=6,learning_rate=0.05,gamma=0.1,colsample_bytree=1.0,random_state=42,tree_method='hist',n_jobs=-1)
v1.fit(X_train,y_train)

/home/yazid/sklearn/.venv/lib64/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:52:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "n_esimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1.0
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [22]:
from sklearn.metrics import root_mean_squared_error
y_pred_1=v1.predict(X_test)
rmse_1=root_mean_squared_error(np.expm1(y_test),np.expm1(y_pred_1))
rmse_1

56.50721604223677

In [23]:
feature_importance(v1)

,Feature,Importance
0,Previous_hour_cnt,0.635666
1,hr,0.120335
2,hr_cos,0.046271
3,Previous_day_cnt,0.036559
4,workingday,0.024858
5,Rush_Hour,0.018594
6,cnt_roll_24_std,0.012175
7,cnt_roll_3h_mean,0.009132
8,week_cos,0.008594
9,temp,0.008176


We drop features with no predictive value :They just introduced noise:

In [24]:
df.drop(['bad_weather','Off_Peak'],axis=1,inplace=True)

In [32]:
X=df.drop(['log_cnt','dteday','casual','registered'],axis=1)
y=df['log_cnt']
X_train=X[:13884]
X_test=X[13884:]
y_train=y[:13884]
y_test=y[13884:]

In [33]:
v2=XGBRegressor(n_esimators=300,subsample=0.8,max_depth=6,learning_rate=0.05,gamma=0.1,colsample_bytree=1.0,random_state=42,tree_method='hist',n_jobs=-1)
v2.fit(X_train,y_train)

/home/yazid/sklearn/.venv/lib64/python3.14/site-packages/xgboost/training.py:200: UserWarning: [18:03:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "n_esimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1.0
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [34]:
y_pred_2=v2.predict(X_test)
rmse_2=root_mean_squared_error(np.expm1(y_test),np.expm1(y_pred_2))
rmse_2

55.69565242716538

In [35]:
feature_importance(v2)

,Feature,Importance
0,Previous_hour_cnt,0.635853
1,hr,0.121255
2,hr_cos,0.046261
3,Previous_day_cnt,0.037806
4,workingday,0.024381
5,Rush_Hour,0.017218
6,cnt_roll_24_std,0.012733
7,cnt_roll_3h_mean,0.008642
8,weathersit,0.008230
9,hr_sin,0.007969
